# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Clustering**:
because it deals with grouping content into their possible performance archetypes.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Prediction**:
The possible content archetypes.
it comes from an observed outcome. after content is clustered, we  define each group

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Silhouette Score**:\
The average distance of page from it's cluster compared to the nearest other clusters. scale -1 to 1(the bigger the better
)\

Thresholds:\
\>0.5 means **good**. clusters have real separation.\
0.25 - 0.49 means **weak**. but could still be useful.\
\< 0.25 means **bad** don't trust. means they're probably arbitrary slices




## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
from datasets import load_dataset
cols_needed = [ 'content_hash_id', 'report_date']
ds_march = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=hf_token
)

df_march = ds_march.select_columns(cols_needed).to_pandas()
print(df_march.shape)
print(df_march['report_date'].min(), df_march['report_date'].max())

df_march_slim = df_march[['report_date', 'content_hash_id']]

daily_check = df_march_slim.groupby('report_date')['content_hash_id'].agg(
    total_rows='count',
    unique_ids='nunique'
).reset_index()

mismatches = daily_check[daily_check['total_rows'] != daily_check['unique_ids']]
print(f"Dates with duplicate content_hash_id: {len(mismatches)}")

print("Therefore we have one row per content(page), for each day on the month")

(9841378, 2)
2026-03-01 2026-03-31
Dates with duplicate content_hash_id: 0
Therefore we have one row per content(page), for each day on the month


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

- The archetypes are defined by interactions across many dimensions at once, not one threshold.
- You don't actually know where the thresholds should go and the data doesn't have one obvious cut point.
- Rules don't scale or update they're a snapshot of your assumptions, not the data.
- The boundaries are almost certainly not the same shape for every page type.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.